In [1]:
import torch
from architectures.flb_transformer import FLB_Transformer
import torch.optim as optim
import torch.nn.functional as F

device = 'cuda'

In [2]:
# Initialize the model and move it to the GPU
model = FLB_Transformer(hidden_dim=256, num_layers=4, seq_len=32, batch_size=2).to(device)

# Create dummy text data matching the config shape
dummy_input = torch.randn(2, 32, 256, device=device, requires_grad=True)

print("Running forward pass...")
y_text, aux_loss = model(dummy_input)

print(f"Output Text Shape {y_text.shape}")
print(f"Auxiliary Loss {aux_loss.item()}")

Running forward pass...
Output Text Shape torch.Size([2, 32, 256])
Auxiliary Loss 1.8229224681854248


In [3]:
# Create a standard optimizer to update the weights
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Create a fake target tensor that matches our expected output shape
dummy_target = torch.randn(2, 32, 256, device=device)

# Clear out any old memory in the optimizer
optimizer.zero_grad()

# Run the forward pass to get our predictions and our internal error
y_text, aux_loss = model(dummy_input)

# Calculate how far off our final prediction was from the fake target
primary_loss = F.mse_loss(y_text, dummy_target)

# Add them together so the network learns to predict the text AND its own lateral states
total_loss = primary_loss + aux_loss

print(f"Total combined loss {total_loss.item()}")

# Send the combined error signal backward through our custom Triton engine
total_loss.backward()

# Look at the lateral projection weights in the very first layer of the grid
first_layer_grad = model.layers[0].proj_lateral.weight.grad

if first_layer_grad is not None:
    print("Success. Gradients safely reached the start of the network.")
    print(f"Gradient magnitude {first_layer_grad.abs().mean().item()}")
else:
    print("The gradient tape snapped somewhere.")

# Tell the optimizer to adjust the weights
optimizer.step()

Total combined loss 3.8256044387817383
Success. Gradients safely reached the start of the network.
Gradient magnitude 0.0005770038696937263
